# NLP Exercises (Part 2)

We have 2 exercises in this section. The exercises are:

4. Build your own Bag Of Words implementation using tokenizer created before.
5. Build a 5-gram model and clean up the results.

## Exercise 4. Build your own Bag Of Words implementation using tokenizer created before 

You need to implement following methods:

- ``fit_transform`` - gets a list of strings and returns matrix with it's BoW representation
- ``get_features_names`` - returns list of words corresponding to columns in BoW

In [1]:
import numpy as np
import spacy

class BagOfWords:
    """Basic BoW implementation."""
    
    __nlp = spacy.blank("en")
    __bow_list = []
    
    # your code goes maybe also here    
    def __tokenize(self, text: str) -> list:
        return [
            token.text.lower()
            for token in self.__nlp(text)
            if token.is_alpha
        ]
    
    def fit_transform(self, corpus: list):
        """Transform list of strings into BoW array.

        Parameters
        ----------
        corpus: List[str]
                Corpus of texts to be transforrmed

        Returns
        -------
        np.array
                Matrix representation of BoW

        """
        # your code goes here
        tokenized_corpus = [self.__tokenize(text) for text in corpus]
        
        self.__bow_list = sorted(
            set(token for document in tokenized_corpus for token in document)
        )
        
        word_to_index = {
            word: index
            for index, word in enumerate(self.__bow_list)
        }
        
        bow_matrix = np.zeros(
            (len(corpus), len(self.__bow_list)),
            dtype=int
        )
        
        for row_index, document in enumerate(tokenized_corpus):
            for token in document:
                column_index = word_to_index[token]
                bow_matrix[row_index, column_index] += 1
        
        return bow_matrix
      

    def get_feature_names(self) -> list:
        """Return words corresponding to columns of matrix.

        Returns
        -------
        List[str]
                Words being transformed by fit function

        """   
        # your code goes here
        return self.__bow_list


corpus = [
     'Bag Of Words is based on counting',
     'words occurences throughout multiple documents.',
     'This is the third document.',
     'As you can see most of the words occur only once.',
     'This gives us a pretty sparse matrix, see below. Really, see below',
]    
    
vectorizer = BagOfWords()

X = vectorizer.fit_transform(corpus)
print(X)

print(vectorizer.get_feature_names())
print(len(vectorizer.get_feature_names()))

[[0 0 1 1 0 0 1 0 0 0 1 0 0 0 0 0 1 1 0 0 0 0 0 0 0 0 0 0 0 1 0]
 [0 0 0 0 0 0 0 0 1 0 0 0 0 1 0 1 0 0 0 0 0 0 0 0 0 0 0 1 0 1 0]
 [0 0 0 0 0 0 0 1 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 1 1 1 0 0 0 0]
 [0 1 0 0 0 1 0 0 0 0 0 0 1 0 1 0 1 0 1 1 0 0 1 0 1 0 0 0 0 1 1]
 [1 0 0 0 2 0 0 0 0 1 0 1 0 0 0 0 0 0 0 0 1 1 2 1 0 0 1 0 1 0 0]]
['a', 'as', 'bag', 'based', 'below', 'can', 'counting', 'document', 'documents', 'gives', 'is', 'matrix', 'most', 'multiple', 'occur', 'occurences', 'of', 'on', 'once', 'only', 'pretty', 'really', 'see', 'sparse', 'the', 'third', 'this', 'throughout', 'us', 'words', 'you']
31


## Exercise 5. Build a 5-gram model and clean up the results.

There are three tasks to do:
1. Use 5-gram model instead of 3.
2. Change to capital letter each first letter of a sentence.
3. Remove the whitespace between the last word in a sentence and . ! or ?.

Hint: for 2. and 3. implement a function called ``clean_generated()`` that takes the generated text and fix both issues at once. It could be easier to fix the text after it's generated rather then doing some changes in the while loop.

In [2]:
import nltk

nltk.download("book")

[nltk_data] Downloading collection 'book'
[nltk_data]    | 
[nltk_data]    | Downloading package abc to
[nltk_data]    |     C:\Users\piotr\AppData\Roaming\nltk_data...
[nltk_data]    |   Unzipping corpora\abc.zip.
[nltk_data]    | Downloading package brown to
[nltk_data]    |     C:\Users\piotr\AppData\Roaming\nltk_data...
[nltk_data]    |   Unzipping corpora\brown.zip.
[nltk_data]    | Downloading package chat80 to
[nltk_data]    |     C:\Users\piotr\AppData\Roaming\nltk_data...
[nltk_data]    |   Unzipping corpora\chat80.zip.
[nltk_data]    | Downloading package cmudict to
[nltk_data]    |     C:\Users\piotr\AppData\Roaming\nltk_data...
[nltk_data]    |   Unzipping corpora\cmudict.zip.
[nltk_data]    | Downloading package conll2000 to
[nltk_data]    |     C:\Users\piotr\AppData\Roaming\nltk_data...
[nltk_data]    |   Unzipping corpora\conll2000.zip.
[nltk_data]    | Downloading package conll2002 to
[nltk_data]    |     C:\Users\piotr\AppData\Roaming\nltk_data...
[nltk_data]    |   U

True

In [3]:
from nltk.book import *

import re
import random

wall_street = text7.tokens

tokens = wall_street

def cleanup():
    compiled_pattern = re.compile("^[a-zA-Z0-9.!?]")
    clean = list(filter(compiled_pattern.match, tokens))
    clean = [token.lower() for token in clean]
    return clean

tokens = cleanup()

def build_ngrams():
    ngrams = []
    for i in range(len(tokens) - N + 1):
        ngrams.append(tokens[i:i + N])
    return ngrams

def ngram_freqs(ngrams):
    counts = {}

    for ngram in ngrams:
        token_seq = SEP.join(ngram[:-1])
        last_token = ngram[-1]

        if token_seq not in counts:
            counts[token_seq] = {}

        if last_token not in counts[token_seq]:
            counts[token_seq][last_token] = 0

        counts[token_seq][last_token] += 1

    return counts

def next_word(text, N, counts):
    token_seq = SEP.join(text.split()[-(N - 1):])

    if token_seq not in counts:
        token_seq = random.choice(list(counts.keys()))

    choices = counts[token_seq].items()

    total = sum(weight for choice, weight in choices)
    r = random.uniform(0, total)

    upto = 0
    for choice, weight in choices:
        upto += weight
        if upto > r:
            return choice

    assert False

*** Introductory Examples for the NLTK Book ***
Loading text1, ..., text9 and sent1, ..., sent9
Type the name of the text or sentence to view it.
Type: 'texts()' or 'sents()' to list the materials.
text1: Moby Dick by Herman Melville 1851
text2: Sense and Sensibility by Jane Austen 1811
text3: The Book of Genesis
text4: Inaugural Address Corpus
text5: Chat Corpus
text6: Monty Python and the Holy Grail
text7: Wall Street Journal
text8: Personals Corpus
text9: The Man Who Was Thursday by G . K . Chesterton 1908


In [4]:
def clean_generated(text):
    text = re.sub(r"\s+([.!?])", r"\1", text)
    text = re.sub(r"\s+", " ", text).strip()

    text = re.sub(
        r"(^|[.!?]\s+)([a-z])",
        lambda match: match.group(1) + match.group(2).upper(),
        text
    )

    return text


N = 5
SEP = " "

sentence_count = 5

ngrams = build_ngrams()

start_seq = None

counts = ngram_freqs(ngrams)

if start_seq is None:
    start_seq = random.choice(list(counts.keys()))

generated = start_seq

sentences = 0
while sentences < sentence_count:
    generated += SEP + next_word(generated, N, counts)
    sentences += 1 if generated.endswith((".", "!", "?")) else 0

generated = clean_generated(generated)

print(generated)

First foldable silicone lens available for cataract surgery. The len foldability enables it to be inserted in smaller incisions than are now possible for cataract surgery the eye care and skin care concern said 0. Cataracts refer to a clouding of the eye natural lens. A man from the bush administration came before the house agriculture committee yesterday to talk about the u.s. Intention to send some 100 million in food aid to poland with more 0 to come from the ec.
